# 📚 Smart Library — Couche Gold
## Notebook 03 : Analytics, Business Intelligence et Time Travel Iceberg

---

### Objectif
Construire les tables **analytiques prêtes pour la visualisation** et démontrer
les fonctionnalités avancées d'Apache Iceberg.

### Tables Gold produites
| Table Gold | Contenu | Cas d'usage |
|---|---|---|
| `gold.top_livres` | Livres les plus vendus + score tendance | Dashboard ventes |
| `gold.ventes_periode` | Agrégations par période/genre/pays | Analyse temporelle |
| `gold.comportement_utilisateurs` | RFM — Récence, Fréquence, Montant | Segmentation clients |
| `gold.historique_ventes` | Ventes enrichies pour Time Travel | Audit & traçabilité |
| `gold.tendances` | KPIs business globaux | Dashboard direction |

### Fonctionnalités Iceberg démontrées
- **Time Travel** : lire les données telles qu'elles étaient à un `snapshot_id` précis
  (`VERSION AS OF`) — équivalent de `as.of.instant` chez Hudi
- **Rollback logique** : comparer deux snapshots pour auditer une correction
- **Snapshots & metadata tables** : `{table}.snapshots`, `{table}.history`

---
## Étape 0 — Initialisation SparkSession

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-Gold")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

# ── Tables Silver (lecture) ────────────────────────────────────
SILVER_COMMANDES    = f"{NS_SILVER}.commandes"
SILVER_LIVRES       = f"{NS_SILVER}.livres"
SILVER_UTILISATEURS = f"{NS_SILVER}.utilisateurs"
SILVER_STOCK        = f"{NS_SILVER}.stock"

# ── Tables Gold (écriture) ─────────────────────────────────────
GOLD_TOP_LIVRES     = f"{NS_GOLD}.top_livres"
GOLD_VENTES_PERIODE = f"{NS_GOLD}.ventes_periode"
GOLD_COMPORTEMENT   = f"{NS_GOLD}.comportement_utilisateurs"
GOLD_HISTORIQUE     = f"{NS_GOLD}.historique_ventes"
GOLD_TENDANCES      = f"{NS_GOLD}.tendances"

from pyspark.sql.window import Window
from datetime import timedelta

print(f"   Source      : {NS_SILVER}")
print(f"   Destination : {NS_GOLD}")

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)
   Source      : lakehouse.silver
   Destination : lakehouse.gold


---
## Étape 1 — Chargement des tables Silver

In [2]:
# ── Lecture de toutes les tables Silver ───────────────────────
# (pas de colonnes techniques à retirer : les métadonnées Iceberg
#  vivent dans le catalogue, pas dans les lignes)
df_commandes = spark.table(SILVER_COMMANDES)
df_livres    = spark.table(SILVER_LIVRES)
df_users     = spark.table(SILVER_UTILISATEURS)
df_stock     = spark.table(SILVER_STOCK)

# Enregistrer comme vues SQL
df_commandes.createOrReplaceTempView("commandes")
df_livres.createOrReplaceTempView("livres")
df_users.createOrReplaceTempView("utilisateurs")
df_stock.createOrReplaceTempView("stock")

print("✅ Tables Silver chargées")
print(f"   Commandes    : {df_commandes.count():,}")
print(f"   Livres       : {df_livres.count():,}")
print(f"   Utilisateurs : {df_users.count():,}")
print(f"   Stock        : {df_stock.count():,}")

✅ Tables Silver chargées
   Commandes    : 5,000
   Livres       : 500
   Utilisateurs : 1,000
   Stock        : 500


---
## Étape 2 — Gold : Top Livres
### Score de tendance = ventes récentes pondérées + note moyenne + popularité

In [3]:
# ── Calcul des ventes par livre ───────────────────────────────
df_ventes_livre = spark.sql("""
    SELECT
        c.livre_id,
        c.titre,
        c.genre,
        c.auteur,
        c.editeur,
        c.categorie_prix,
        COUNT(*)                                    AS nb_commandes,
        SUM(c.quantite)                             AS nb_exemplaires_vendus,
        ROUND(SUM(c.montant_total), 2)              AS chiffre_affaires,
        ROUND(AVG(c.montant_total), 2)              AS panier_moyen,
        COUNT(DISTINCT c.utilisateur_id)            AS nb_clients_distincts,
        -- Ventes des 30 derniers jours (tendance récente)
        SUM(CASE
            WHEN DATEDIFF(CURRENT_DATE(), TO_DATE(date_commande)) <= 30
            THEN quantite ELSE 0 END)               AS ventes_30j,
        -- Ventes des 90 derniers jours
        SUM(CASE
            WHEN DATEDIFF(CURRENT_DATE(), TO_DATE(date_commande)) <= 90
            THEN quantite ELSE 0 END)               AS ventes_90j
    FROM commandes c
    WHERE c.statut IN ('livré', 'en_cours')
    GROUP BY c.livre_id, c.titre, c.genre, c.auteur,
             c.editeur, c.categorie_prix
""")

print(f"📊 Livres avec ventes : {df_ventes_livre.count()}")

📊 Livres avec ventes : 500


In [4]:
# ── Calcul du score de tendance ───────────────────────────────
# Formule : (ventes_30j * 3 + ventes_90j * 2 + nb_commandes) / 6
# Pondération plus forte sur les ventes récentes

df_top_livres = (
    df_ventes_livre
    .join(
        df_livres.select("livre_id", "note_moyenne", "nb_avis",
                         "score_popularite", "prix_vente", "marge_pct"),
        on="livre_id", how="left"
    )
    # Score de tendance pondéré
    .withColumn("score_tendance",
                F.round(
                    (F.col("ventes_30j") * 3 +
                     F.col("ventes_90j") * 2 +
                     F.col("nb_commandes")) / 6.0, 2
                ))
    # Rang global
    .withColumn("rang_ventes",
                F.rank().over(
                    Window.orderBy(F.col("nb_exemplaires_vendus").desc())
                ))
    # Rang par genre
    .withColumn("rang_dans_genre",
                F.rank().over(
                    Window.partitionBy("genre")
                          .orderBy(F.col("nb_exemplaires_vendus").desc())
                ))
    # Catégorie de performance
    .withColumn("performance",
                F.when(F.col("rang_ventes") <= 10,  F.lit("Bestseller"))
                 .when(F.col("rang_ventes") <= 50,  F.lit("Populaire"))
                 .when(F.col("rang_ventes") <= 150, F.lit("Correct"))
                 .otherwise(F.lit("Faible")))
    .withColumn("updated_at",    F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_genre", F.col("genre"))
)

print("🏆 TOP 10 LIVRES — Toutes catégories")
print("=" * 70)
(
    df_top_livres
    .orderBy("rang_ventes")
    .select("rang_ventes", "titre", "genre", "nb_exemplaires_vendus",
            "chiffre_affaires", "note_moyenne", "score_tendance", "performance")
    .show(10, truncate=True)
)

print("\n🏆 TOP 5 PAR GENRE")
(
    df_top_livres
    .filter(F.col("rang_dans_genre") <= 3)
    .orderBy("genre", "rang_dans_genre")
    .select("genre", "rang_dans_genre", "titre", "nb_exemplaires_vendus", "score_tendance")
    .show(20, truncate=True)
)

🏆 TOP 10 LIVRES — Toutes catégories
+-----------+--------------------+--------------------+---------------------+----------------+------------+--------------+-----------+
|rang_ventes|               titre|               genre|nb_exemplaires_vendus|chiffre_affaires|note_moyenne|score_tendance|performance|
+-----------+--------------------+--------------------+---------------------+----------------+------------+--------------+-----------+
|          1|Air Poitrine Refu...|Développement per...|                   69|          1858.3|         1.9|         15.17| Bestseller|
|          2|Hors Marquer Rien...|          Biographie|                   66|         1660.35|         1.1|          9.33| Bestseller|
|          3|Écraser Connaître...|         Philosophie|                   54|          1543.2|         2.9|          3.83| Bestseller|
|          4|Regard Bientôt At...|     Science-fiction|                   53|         1651.56|         2.0|          3.17| Bestseller|
|          5|      

In [5]:
# ── Écriture Iceberg Gold ──────────────────────────────────────
create_or_replace_table(
    df_top_livres, GOLD_TOP_LIVRES,
    partition_cols=["partition_genre"], type_table="copy-on-write"
)

n = spark.table(GOLD_TOP_LIVRES).count()
print(f"✅ {GOLD_TOP_LIVRES} — {n} livres avec scores")

✅ lakehouse.gold.top_livres — 500 livres avec scores


---
## Étape 3 — Gold : Ventes par Période
### Agrégations temporelles pour le dashboard

In [6]:
# ── Agrégations par période, genre et pays ────────────────────
df_ventes_periode = spark.sql("""
    SELECT
        partition_periode                            AS periode,
        annee,
        mois,
        genre,
        pays                                         AS pays_client,
        COUNT(*)                                     AS nb_commandes,
        SUM(quantite)                                AS nb_exemplaires,
        ROUND(SUM(montant_total), 2)                 AS chiffre_affaires,
        ROUND(AVG(montant_total), 2)                 AS panier_moyen,
        COUNT(DISTINCT utilisateur_id)               AS nb_clients_actifs,
        ROUND(SUM(montant_ht), 2)                    AS ca_ht,
        ROUND(SUM(montant_tva), 2)                   AS tva_collectee,
        -- Taux de commandes livrées
        ROUND(
            SUM(CASE WHEN statut = 'livré' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1)                                           AS taux_livraison_pct
    FROM commandes
    GROUP BY partition_periode, annee, mois, genre, pays
    ORDER BY periode, genre
""")

# Ajouter clé primaire et métadonnées
df_ventes_periode = (
    df_ventes_periode
    .withColumn("vente_periode_id",
                F.concat_ws("_", F.col("periode"), F.col("genre"), F.col("pays_client")))
    .withColumn("updated_at",       F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_annee",  F.col("annee"))
)

print(f"📅 Ventes par période : {df_ventes_periode.count()} combinaisons période/genre/pays")
print()
print("Chiffre d'affaires par année :")
spark.sql("""
    SELECT annee,
           COUNT(DISTINCT utilisateur_id)    AS clients_actifs,
           SUM(quantite)                     AS exemplaires_vendus,
           ROUND(SUM(montant_total), 2)      AS ca_total
    FROM commandes
    GROUP BY annee
    ORDER BY annee
""").show()

print("Top genres par chiffre d'affaires :")
spark.sql("""
    SELECT genre,
           ROUND(SUM(montant_total), 2) AS ca_total,
           COUNT(*)                     AS nb_commandes
    FROM commandes
    GROUP BY genre
    ORDER BY ca_total DESC
    LIMIT 5
""").show()

📅 Ventes par période : 955 combinaisons période/genre/pays

Chiffre d'affaires par année :
+-----+--------------+------------------+---------+
|annee|clients_actifs|exemplaires_vendus| ca_total|
+-----+--------------+------------------+---------+
| 2024|           694|              3590|107827.82|
| 2025|           923|              7569| 226316.9|
| 2026|           735|              3863|115676.82|
+-----+--------------+------------------+---------+

Top genres par chiffre d'affaires :
+--------------------+--------+------------+
|               genre|ca_total|nb_commandes|
+--------------------+--------+------------+
|          Biographie|49084.43|         551|
|            Policier|48272.43|         534|
|            Jeunesse| 48204.7|         530|
|Développement per...|48014.53|         519|
|              Poésie|47910.69|         526|
+--------------------+--------+------------+



In [7]:
# ── Écriture Iceberg Gold ──────────────────────────────────────
create_or_replace_table(
    df_ventes_periode, GOLD_VENTES_PERIODE,
    partition_cols=["partition_annee"], type_table="copy-on-write"
)

n = spark.table(GOLD_VENTES_PERIODE).count()
print(f"✅ {GOLD_VENTES_PERIODE} — {n} agrégations")

✅ lakehouse.gold.ventes_periode — 955 agrégations


---
## Étape 4 — Gold : Comportement Utilisateurs (Analyse RFM)

> 💡 **RFM** = Récence + Fréquence + Montant — méthode classique de segmentation client.
> - **Récence** : depuis combien de jours a-t-il commandé ?
> - **Fréquence** : combien de commandes au total ?
> - **Montant** : quel est son panier total ?

In [8]:
# ── Calcul RFM par utilisateur ────────────────────────────────
df_rfm = spark.sql("""
    SELECT
        c.utilisateur_id,
        u.nom,
        u.pays,
        u.segment,
        u.anciennete_jours,
        -- Récence : jours depuis dernière commande
        DATEDIFF(CURRENT_DATE(),
                 MAX(TO_DATE(c.date_commande)))       AS recence_jours,
        -- Fréquence : nb commandes livrées
        COUNT(CASE WHEN c.statut = 'livré'
                   THEN 1 END)                        AS frequence,
        -- Montant total dépensé
        ROUND(SUM(CASE WHEN c.statut IN ('livré', 'en_cours')
                       THEN c.montant_total
                       ELSE 0 END), 2)                AS montant_total,
        -- Panier moyen
        ROUND(AVG(c.montant_total), 2)                AS panier_moyen,
        -- Genres préférés (les plus commandés)
        COUNT(DISTINCT c.genre)                       AS nb_genres_differents,
        -- Délai moyen de livraison
        ROUND(AVG(c.delai_livraison_jours), 1)        AS delai_moyen_jours
    FROM commandes c
    JOIN utilisateurs u ON c.utilisateur_id = u.utilisateur_id
    GROUP BY c.utilisateur_id, u.nom, u.pays, u.segment, u.anciennete_jours
""")

print(f"👥 Profils RFM calculés : {df_rfm.count()} utilisateurs")

👥 Profils RFM calculés : 993 utilisateurs


In [9]:
# ── Scoring RFM (1 à 5 pour chaque dimension) ─────────────────
# Percentiles pour le scoring
percentiles_R = df_rfm.approxQuantile("recence_jours", [0.2, 0.4, 0.6, 0.8], 0.05)
percentiles_F = df_rfm.approxQuantile("frequence",     [0.2, 0.4, 0.6, 0.8], 0.05)
percentiles_M = df_rfm.approxQuantile("montant_total", [0.2, 0.4, 0.6, 0.8], 0.05)

def score_r(col):
    # Récence : plus faible = mieux (acheté récemment)
    return (F.when(col <= percentiles_R[0], 5)
             .when(col <= percentiles_R[1], 4)
             .when(col <= percentiles_R[2], 3)
             .when(col <= percentiles_R[3], 2)
             .otherwise(1))

def score_fm(col, percentiles):
    # Fréquence/Montant : plus élevé = mieux
    return (F.when(col >= percentiles[3], 5)
             .when(col >= percentiles[2], 4)
             .when(col >= percentiles[1], 3)
             .when(col >= percentiles[0], 2)
             .otherwise(1))

df_rfm_score = (
    df_rfm
    .withColumn("score_r", score_r(F.col("recence_jours")))
    .withColumn("score_f", score_fm(F.col("frequence"),     percentiles_F))
    .withColumn("score_m", score_fm(F.col("montant_total"), percentiles_M))
    .withColumn("score_rfm",
                F.col("score_r") + F.col("score_f") + F.col("score_m"))
    # Segment RFM
    .withColumn("segment_rfm",
                F.when((F.col("score_r") >= 4) & (F.col("score_f") >= 4), F.lit("Champion"))
                 .when((F.col("score_r") >= 3) & (F.col("score_m") >= 4), F.lit("Client fidèle"))
                 .when((F.col("score_r") >= 4) & (F.col("score_f") <= 2), F.lit("Nouveau prometteur"))
                 .when((F.col("score_r") <= 2) & (F.col("score_f") >= 4), F.lit("À risque"))
                 .when((F.col("score_r") <= 2) & (F.col("score_f") <= 2), F.lit("Inactif"))
                 .otherwise(F.lit("Ordinaire")))
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_pays",  F.col("pays"))
)

print("✅ Scoring RFM calculé")
print()
print("Distribution des segments RFM :")
df_rfm_score.groupBy("segment_rfm").count().orderBy("count", ascending=False).show()

print("Top 10 Champions (meilleurs clients) :")
(
    df_rfm_score
    .filter(F.col("segment_rfm") == "Champion")
    .orderBy(F.col("score_rfm").desc())
    .select("utilisateur_id", "nom", "pays", "frequence",
            "montant_total", "score_rfm", "segment_rfm")
    .show(10, truncate=True)
)

✅ Scoring RFM calculé

Distribution des segments RFM :
+------------------+-----+
|       segment_rfm|count|
+------------------+-----+
|          Champion|  291|
|          À risque|  216|
|           Inactif|  204|
|     Client fidèle|  105|
|         Ordinaire|  105|
|Nouveau prometteur|   72|
+------------------+-----+

Top 10 Champions (meilleurs clients) :
+--------------+----------+-------+---------+-------------+---------+-----------+
|utilisateur_id|       nom|   pays|frequence|montant_total|score_rfm|segment_rfm|
+--------------+----------+-------+---------+-------------+---------+-----------+
|      USR00960|  Courtois| France|        4|       685.68|       15|   Champion|
|      USR00877|   Ferrand| France|        4|       691.75|       15|   Champion|
|      USR00763|   Jourdan| France|        7|      1020.13|       15|   Champion|
|      USR00283|    Bonnin|  Maroc|        8|       671.03|       15|   Champion|
|      USR00818| Alexandre|Tunisie|        8|       899.91|  

In [10]:
# ── Écriture Iceberg Gold ──────────────────────────────────────
create_or_replace_table(
    df_rfm_score, GOLD_COMPORTEMENT,
    partition_cols=["partition_pays"], type_table="copy-on-write"
)

n = spark.table(GOLD_COMPORTEMENT).count()
print(f"✅ {GOLD_COMPORTEMENT} — {n} profils RFM")

✅ lakehouse.gold.comportement_utilisateurs — 993 profils RFM


---
## Étape 5 — Gold : Historique des Ventes + TIME TRAVEL ICEBERG

> 💡 **Time Travel** = lire les données telles qu'elles étaient à un `snapshot_id` précis dans le passé.
> Cas d'usage réels : audit financier, correction d'erreurs, comparaison avant/après.
>
> Avec Hudi, on référence un état passé par son `_hoodie_commit_time` (`as.of.instant`).
> Avec Iceberg, chaque écriture crée un **snapshot** identifié par un `snapshot_id` (ou un
> horodatage), consultable dans la metadata table `{table}.snapshots` et exploitable via
> `VERSION AS OF <snapshot_id>` ou `TIMESTAMP AS OF <timestamp>` en SQL.

In [11]:
# ── Construction table historique ─────────────────────────────
df_historique = (
    df_commandes
    .filter(F.col("statut").isin(["livré", "annulé", "remboursé"]))
    .withColumn("historique_id",
                F.concat_ws("_", F.col("commande_id"), F.col("statut")))
    .withColumn("annee_mois",
                F.concat_ws("-", F.col("annee"), F.col("mois")))
    .withColumn("est_remboursement",
                F.col("statut").isin(["annulé", "remboursé"]).cast("boolean"))
    .withColumn("impact_ca",
                F.when(F.col("statut") == "livré",
                       F.col("montant_total"))
                 .otherwise(-F.col("montant_total")))
    .withColumn("updated_at",        F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_periode", F.col("partition_periode"))
)

create_or_replace_table(
    df_historique, GOLD_HISTORIQUE,
    partition_cols=["partition_periode"], type_table="copy-on-write"
)

# Sauvegarder le snapshot_id du premier commit
snapshot_v1 = (
    spark.sql(f"SELECT snapshot_id FROM {GOLD_HISTORIQUE}.snapshots ORDER BY committed_at DESC LIMIT 1")
    .first()["snapshot_id"]
)

n = spark.table(GOLD_HISTORIQUE).count()
print(f"✅ {GOLD_HISTORIQUE} — {n} lignes")
print(f"   Snapshot V1 : {snapshot_v1}")

✅ lakehouse.gold.historique_ventes — 4003 lignes
   Snapshot V1 : 3731074348899197005


In [12]:
# ════════════════════════════════════════════════════════════
# DÉMONSTRATION TIME TRAVEL ICEBERG
# Scénario : erreur de prix → correction → lecture historique
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("  DÉMONSTRATION TIME TRAVEL ICEBERG")
print("  Scénario : correction d'une erreur de données")
print("=" * 60)

# État V1 — avant correction
ca_v1 = (
    spark.table(GOLD_HISTORIQUE)
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.sum("impact_ca"), 2).alias("ca_total"))
    .first()["ca_total"]
)
print(f"\n  V1 — CA total (état initial) : {ca_v1} €")

# Simuler une correction : appliquer une remise de 10% sur 20 commandes
df_correction = (
    spark.table(GOLD_HISTORIQUE)
    .limit(20)
    .withColumn("impact_ca",
                F.round(F.col("impact_ca") * 0.9, 2))
    .withColumn("montant_total",
                F.round(F.col("montant_total") * 0.9, 2))
    .withColumn("updated_at",
                F.lit(datetime.now().isoformat()).cast(StringType()))
)

# UPSERT (MERGE INTO) de la correction — crée un nouveau snapshot V2
upsert_table(spark, df_correction, GOLD_HISTORIQUE, key_cols=["historique_id"])

# État V2 — après correction
ca_v2 = (
    spark.table(GOLD_HISTORIQUE)
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.sum("impact_ca"), 2).alias("ca_total"))
    .first()["ca_total"]
)
print(f"  V2 — CA total (après remise 10%) : {ca_v2} €")
print(f"  Différence : {round(ca_v2 - ca_v1, 2)} €")

  DÉMONSTRATION TIME TRAVEL ICEBERG
  Scénario : correction d'une erreur de données

  V1 — CA total (état initial) : 293297.83 €
  V2 — CA total (après remise 10%) : 293177.63 €
  Différence : -120.2 €


In [13]:
# ── TIME TRAVEL : relire l'état V1 (avant correction) ────────
print("=" * 60)
print("  TIME TRAVEL — Lecture à l'état V1")
print(f"  Snapshot de référence : {snapshot_v1}")
print("=" * 60)

# Méthode 1 : via l'option "snapshot-id" du DataFrameReader
df_historique_v1 = (
    spark.read.format("iceberg")
    .option("snapshot-id", snapshot_v1)
    .load(GOLD_HISTORIQUE)
)

# Méthode 2 (équivalente) : via SQL — VERSION AS OF
# df_historique_v1 = spark.sql(f"SELECT * FROM {GOLD_HISTORIQUE} VERSION AS OF {snapshot_v1}")

ca_relu_v1 = (
    df_historique_v1
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.sum("impact_ca"), 2).alias("ca_total"))
    .first()["ca_total"]
)

print(f"\n  CA relu depuis snapshot V1 : {ca_relu_v1} €")
print(f"  CA actuel (V2)             : {ca_v2} €")
print()
print(f"  ✅ Time Travel confirmé :")
print(f"     V1 = {ca_relu_v1} € | V2 = {ca_v2} €")
print(f"     On peut auditer n'importe quel état passé")

# Comparer les 5 premières lignes V1 vs V2
print("\n  5 premières lignes — ÉTAT V1 (avant correction) :")
df_historique_v1.select(
    "historique_id", "montant_total", "impact_ca", "statut"
).show(5, truncate=True)

print("  5 premières lignes — ÉTAT V2 (après correction) :")
spark.table(GOLD_HISTORIQUE).select(
    "historique_id", "montant_total", "impact_ca", "statut"
).show(5, truncate=True)

print("  💡 Historique complet des snapshots :")
snapshots(spark, GOLD_HISTORIQUE).select("snapshot_id", "committed_at", "operation").show(truncate=False)

  TIME TRAVEL — Lecture à l'état V1
  Snapshot de référence : 3731074348899197005

  CA relu depuis snapshot V1 : 293297.83 €
  CA actuel (V2)             : 293177.63 €

  ✅ Time Travel confirmé :
     V1 = 293297.83 € | V2 = 293177.63 €
     On peut auditer n'importe quel état passé

  5 premières lignes — ÉTAT V1 (avant correction) :
+-----------------+-------------+---------+------+
|    historique_id|montant_total|impact_ca|statut|
+-----------------+-------------+---------+------+
| ORD0000222_livré|         39.5|     39.5| livré|
|ORD0000229_annulé|        47.28|   -47.28|annulé|
| ORD0000243_livré|        32.82|    32.82| livré|
|ORD0000288_annulé|        31.57|   -31.57|annulé|
| ORD0000318_livré|        12.36|    12.36| livré|
+-----------------+-------------+---------+------+
only showing top 5 rows

  5 premières lignes — ÉTAT V2 (après correction) :
+----------------+-------------+---------+------+
|   historique_id|montant_total|impact_ca|statut|
+----------------+--------

---
## Étape 6 — Gold : KPIs & Tendances Globales

In [14]:
# ── KPIs business globaux ─────────────────────────────────────
df_kpis = spark.sql("""
    SELECT
        'global'                                      AS perimetre,
        COUNT(DISTINCT utilisateur_id)                AS nb_clients_total,
        COUNT(*)                                      AS nb_commandes_total,
        ROUND(SUM(montant_total), 2)                  AS ca_total,
        ROUND(AVG(montant_total), 2)                  AS panier_moyen_global,
        ROUND(SUM(CASE WHEN statut='livré'
                       THEN montant_total ELSE 0 END), 2) AS ca_livre,
        ROUND(
            SUM(CASE WHEN statut='livré' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1)                                            AS taux_livraison_pct,
        ROUND(
            SUM(CASE WHEN statut='annulé' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1)                                            AS taux_annulation_pct,
        COUNT(DISTINCT genre)                         AS nb_genres_vendus
    FROM commandes
""")

# KPIs par pays
df_kpis_pays = spark.sql("""
    SELECT
        pays                                          AS perimetre,
        COUNT(DISTINCT utilisateur_id)                AS nb_clients_total,
        COUNT(*)                                      AS nb_commandes_total,
        ROUND(SUM(montant_total), 2)                  AS ca_total,
        ROUND(AVG(montant_total), 2)                  AS panier_moyen_global,
        ROUND(SUM(CASE WHEN statut='livré'
                       THEN montant_total ELSE 0 END), 2) AS ca_livre,
        ROUND(
            SUM(CASE WHEN statut='livré' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1)                                            AS taux_livraison_pct,
        ROUND(
            SUM(CASE WHEN statut='annulé' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1)                                            AS taux_annulation_pct,
        COUNT(DISTINCT genre)                         AS nb_genres_vendus
    FROM commandes
    GROUP BY pays
    ORDER BY ca_total DESC
""")

df_tendances = (
    df_kpis.union(df_kpis_pays)
    .withColumn("tendance_id",  F.col("perimetre"))
    .withColumn("updated_at",   F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("partition_p",  F.col("perimetre"))
)

print("📊 KPIs GLOBAUX")
print("=" * 65)
df_kpis.show(truncate=False)

print("\n📊 KPIs PAR PAYS")
df_kpis_pays.show(truncate=False)

📊 KPIs GLOBAUX
+---------+----------------+------------------+---------+-------------------+---------+------------------+-------------------+----------------+
|perimetre|nb_clients_total|nb_commandes_total|ca_total |panier_moyen_global|ca_livre |taux_livraison_pct|taux_annulation_pct|nb_genres_vendus|
+---------+----------------+------------------+---------+-------------------+---------+------------------+-------------------+----------------+
|global   |993             |5000              |449821.54|89.96              |293297.83|65.4              |9.7                |10              |
+---------+----------------+------------------+---------+-------------------+---------+------------------+-------------------+----------------+


📊 KPIs PAR PAYS
+---------+----------------+------------------+---------+-------------------+---------+------------------+-------------------+----------------+
|perimetre|nb_clients_total|nb_commandes_total|ca_total |panier_moyen_global|ca_livre |taux_livraison_p

In [15]:
# ── Écriture Iceberg Gold ──────────────────────────────────────
create_or_replace_table(
    df_tendances, GOLD_TENDANCES,
    partition_cols=["partition_p"], type_table="copy-on-write"
)

n = spark.table(GOLD_TENDANCES).count()
print(f"✅ {GOLD_TENDANCES} — {n} KPIs")

✅ lakehouse.gold.tendances — 5 KPIs


---
## Étape 7 — Bilan de la couche Gold

In [16]:
# ── Résumé complet ────────────────────────────────────────────
tables_gold = [
    ("gold.top_livres",               GOLD_TOP_LIVRES),
    ("gold.ventes_periode",           GOLD_VENTES_PERIODE),
    ("gold.comportement_utilisateurs",GOLD_COMPORTEMENT),
    ("gold.historique_ventes",        GOLD_HISTORIQUE),
    ("gold.tendances",                GOLD_TENDANCES),
]

print("=" * 58)
print(f"  {'TABLE GOLD':<34} {'LIGNES':>8}")
print("=" * 58)
total = 0
for nom, table in tables_gold:
    try:
        n = spark.table(table).count()
        total += n
        print(f"  {nom:<34} {n:>8,}  ✅")
    except Exception as e:
        print(f"  {nom:<34} {'ERREUR':>8}  ❌")
print("=" * 58)
print(f"  {'TOTAL':<34} {total:>8,}")
print("=" * 58)
print()
print("  Fonctionnalités Iceberg démontrées :")
print("  ✅ Time Travel  — lecture à snapshot_v1 (état avant correction)")
print("  ✅ UPSERT Gold  — correction de 20 lignes via MERGE INTO")
print("  ✅ Analyse RFM  — segmentation Champion/Fidèle/À risque")
print("  ✅ Score tendance — pondération ventes récentes/historiques")
print()
print("  MinIO Console  → http://localhost:9001")
print("  Bucket         → lakehouse/gold/")
print("  Catalogue REST → http://localhost:8181 (Lakekeeper)")
print()
print("  ➡  Prochain : 04_streaming_pipeline.ipynb")
print("     Spark Structured Streaming → Iceberg merge-on-read")

  TABLE GOLD                           LIGNES
  gold.top_livres                         500  ✅
  gold.ventes_periode                     955  ✅
  gold.comportement_utilisateurs          993  ✅
  gold.historique_ventes                4,003  ✅
  gold.tendances                            5  ✅
  TOTAL                                 6,456

  Fonctionnalités Iceberg démontrées :
  ✅ Time Travel  — lecture à snapshot_v1 (état avant correction)
  ✅ UPSERT Gold  — correction de 20 lignes via MERGE INTO
  ✅ Analyse RFM  — segmentation Champion/Fidèle/À risque
  ✅ Score tendance — pondération ventes récentes/historiques

  MinIO Console  → http://localhost:9001
  Bucket         → lakehouse/gold/
  Catalogue REST → http://localhost:8181 (Lakekeeper)

  ➡  Prochain : 04_streaming_pipeline.ipynb
     Spark Structured Streaming → Iceberg merge-on-read


In [17]:
spark.stop()
print("✅ SparkSession fermée — Couche Gold complète !")

✅ SparkSession fermée — Couche Gold complète !
